# Completed SO4 example — real notebook execution

This is the recorded completed execution for public TCGA case **TCGA-OR-A5JI**, dated 2026-11-04.
`PATHXAI_VERSION` was set to `SO4`. This run used checksum-verified local D1/SAM3/CONCH assets
and 5 already downloaded, checksum-verified GDC diagnostic slides; it did not test downloading our D1 from HF.
SO4 also used the authorized local KEEP files. The configured vision model was Qwen3.6-27B-Q6_K.
The anatomical site supplied before inference was **adrenal gland / adrenocortical tissue**.
No reference diagnosis, grade or reference report was provided to the inference pipeline.

All code cells and their execution counts are retained from the successful notebook. For publication,
ANSI formatting and repetitive heartbeat lines were removed, feature progress bars were condensed
to their recorded completion lines, adjacent stdout records were combined,
and machine-specific paths were replaced with relative example links or labeled local-path placeholders.
Warning messages and inference outputs are retained; no inference output was fabricated.

Read the outputs below without running anything. The final report text is included at the end.
[HTML report](../SO4/report.html) · [Saved text](../SO4/generated_long_report.txt)

GitHub previews this notebook. To view the HTML as a page, use the repository's GitHub Pages site
or download the full examples folder and open the report locally. To run your own case, use the
clean [main notebook](../../PathXAI_report.ipynb), not this archived execution record.

Completion does not imply every generated clinical statement is correct. Read the
[result comparison](../../docs/RESULT_COMPARISON.md) for the historical comparison and known discrepancies.


# PathXAI: a TCGA patient to an HTML report

Run this notebook on your own NVIDIA GPU computer/server after following [installation](docs/INSTALLATION.md).
Choose SO1/Core or SO4/Full; both use D1 segmentation internally. This is real inference, not a stored-report viewer.
The public sample is the patient-level diagnostic slide set for TCGA-OR-A5JI, with a documented
anatomical site supplied before inference. All downloaded slides contribute to one patient candidate
pool and one report. The exact slide inventory, sizes and checksums are pinned in `configs/sample_tcga.json`.
The workflow matches the historical patient input scope; stochastic API generation need not reproduce exact wording.

Obtain your own access to SAM3, CONCH, and PathXAI D1 on Hugging Face. SO4 additionally needs KEEP access.
Supply a vision-capable Qwen API endpoint/model and credentials. Selected ROI images are sent to that API.
Never save credentials in notebook cells or outputs. References are used only in the optional evaluation cell after generation.


In [1]:
from pathlib import Path
import os, sys, json
from getpass import getpass

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pathxai_tutorial').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Start Jupyter from the PathXAI_tutorial repository directory.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
from pathxai_tutorial import download_patient, sample_metadata, make_config, check_api, run_report
from pathxai_tutorial.models import download_models
print('Tutorial:', ROOT.name)


Tutorial: PathXAI_tutorial


## 1. Choose a version and configure paths

Set the segmentation interpreter to the `pathxai-segmentation-py312` environment from the installation guide.
The current notebook kernel runs the report/CONCH pipeline. The D1 repository is `RayTan/PathXAI-Seg`;
fill in its published exact commit hash after obtaining access. The revision must be provided before remote download can work.
The optional local model configuration is documented in [the inference guide](docs/INFERENCE.md).


In [2]:
VERSION = os.environ.get('PATHXAI_VERSION', 'SO1')  # SO1 or SO4
SEGMENTATION_PYTHON = os.environ.get('PATHXAI_SEG_PYTHON', '/path/to/pathxai-segmentation-py312/bin/python')
D1_REPO_ID = os.environ.get('PATHXAI_D1_REPO', 'RayTan/PathXAI-Seg')
D1_REVISION = os.environ.get('PATHXAI_D1_REVISION', '')  # immutable 40-character HF commit
QWEN_API_BASE = os.environ.get('QWEN_API_BASE', '')
QWEN_MODEL = os.environ.get('QWEN_MODEL', '')
MAX_SECONDS = int(os.environ.get('PATHXAI_MAX_SECONDS', '3600'))
LOCAL_MODEL_CONFIG = os.environ.get('PATHXAI_LOCAL_MODEL_CONFIG', '')

# This optional JSON file contains paths and endpoint/model settings, never keys.
local = json.loads(Path(LOCAL_MODEL_CONFIG).read_text()) if LOCAL_MODEL_CONFIG else {}
if 'api_key' in local or 'hf_token' in local:
    raise ValueError('Remove credentials from the local configuration; use environment variables.')


## 2. Credentials and model downloads

The adapter download contains our LoRA, learned prompts and necessary D1 auxiliary parameters.
SAM3 base weights and CONCH come from their original repositories under your own permissions.
No automatic model substitution is performed when access is unavailable.


In [3]:
api_key = os.environ.get('QWEN_API_KEY', '')
if not api_key and os.environ.get('PATHXAI_API_NO_KEY', '') != '1':
    api_key = getpass('Qwen API key (leave blank only if your endpoint needs none): ')

if local:
    config = make_config(**local, api_key=api_key)
else:
    if not D1_REPO_ID or not D1_REVISION:
        raise ValueError('Set D1_REPO_ID and D1_REVISION from the model release before downloading.')
    hf_token = os.environ.get('HF_TOKEN') or getpass('HF read token for your approved model access: ')
    assets = download_models(d1_repo_id=D1_REPO_ID, d1_revision=D1_REVISION,
                             version=VERSION, token=hf_token)
    config = make_config(segmentation_python=SEGMENTATION_PYTHON,
                         qwen_api=QWEN_API_BASE, qwen_model=QWEN_MODEL,
                         api_key=api_key, hf_token=hf_token, **assets)
print('Selected report version:', VERSION)


Selected report version: SO4


## 3. Download and validate the public TCGA patient

The downloader checks every pinned GDC record, file size, MD5 and SHA-256. Interrupted transfers can be resumed.
If you already have the slide set, set `PATHXAI_SAMPLE_DIR` to its parent directory (nested GDC UUID folders
are supported). Every expected slide must pass verification; an incomplete patient is never run silently.
For your own data, replace `slides`, `case_id` and `known_site` with your patient's slide list, identifier
and anatomical sampling site. Do not pass a tumor diagnosis as `known_site`.


In [4]:
sample = sample_metadata()
slides = download_patient(ROOT / 'data', existing_dir=os.environ.get('PATHXAI_SAMPLE_DIR') or None)
case_id = sample['case_id']
known_site = sample['known_site']
print('Patient:', case_id)
print('Known anatomical site:', known_site)
print('Verified slides:', len(slides))
for slide in slides:
    print(' -', slide.name)


Patient: TCGA-OR-A5JI
Known anatomical site: adrenal gland / adrenocortical tissue
Verified slides: 5
 - TCGA-OR-A5JI-01Z-00-DX1.E133042C-62B2-4015-853B-436830988A53.svs
 - TCGA-OR-A5JI-01Z-00-DX2.CEEA791D-1475-4D2B-8E8D-9EB44D42C937.svs
 - TCGA-OR-A5JI-01Z-00-DX3.A0770B66-D219-4908-BB9B-32790497CFFE.svs
 - TCGA-OR-A5JI-01Z-00-DX4.367BF4F7-454E-4954-B9C1-4CCF7B7CD52E.svs
 - TCGA-OR-A5JI-01Z-00-DX5.4F5C6E00-2B26-432A-9C36-6C810AF0C7ED.svs


## 4. Verify the configured API

This makes one small synthetic-image request. A successful probe checks connectivity only; it does not
guarantee the full scientific workflow will succeed. No sample WSI is sent in this step.


In [5]:
probe = check_api(config)
print(probe['message'])
if not probe['ok']:
    raise RuntimeError('API probe failed. Correct endpoint, model and key before running inference.')


Synthetic image request and JSON response verified.


## 5. Run report generation

One case and one version run at a time. Keep the kernel/server alive; progress is also written to disk.
The time budget cancels the owned worker and preserves partial artifacts. Re-running uses a new output directory.
The sample supplies its documented anatomical site explicitly, as the historical patient run did.
Only slide paths, patient identifier and anatomical site are passed to inference; cancer-project labels,
reference diagnosis, grade and evaluation results are not supplied to the report generator.


In [6]:
result = run_report(config, slide_paths=slides, case_id=case_id, version=VERSION,
                    known_site=known_site, max_seconds=MAX_SECONDS,
                    output_dir=os.environ.get('PATHXAI_OUTPUT_DIR') or None)
print('Report:', result['report_html'])
print('Report text:', result['report_text'])


[preflight] Checking engine configuration
[asset_preflight] Verifying pinned model asset identities
[wsi_preparation] Resolving candidate ROIs
[wsi_preparation] Preparing slide 1/5 with original CONCH ranking
[wsi_preparation] Preparing slide 2/5 with original CONCH ranking
[wsi_preparation] Preparing slide 3/5 with original CONCH ranking
[wsi_preparation] Preparing slide 4/5 with original CONCH ranking
[wsi_preparation] Preparing slide 5/5 with original CONCH ranking
[engine] Starting original FULL pipeline (SO4_grade_aux_keep)
[engine_log]     [1/10]   ROI evidence seeding (20 candidates) ...
[engine_log]     [1/10] → 3 seed ROIs selected (TCGA-OR-A5JI-01Z-00-DX1.E133042C-62B2-4015-853B-436830988A53_foundation_x59277_y40483_l0_w4096_h4096, TCGA-OR-A5JI-01Z-00-DX1.E133042C-62B2-4015-853B-436830988A53_foundation_x53199_y48587_l0_w4096_h4096, TCGA-OR-A5JI-01Z-00-DX2.CEEA791D-1475-4D2B-8E8D-9EB44D42C937_foundation_x68821_y16171_l0_w4096_h4096)
[engine_log]     [2/10]   Site: known adrena

## 6. Open/export the report

Keep the HTML together with its `assets/` directory. Download/copy the whole portable folder to share the report.
The raw scientific outputs and exact configuration/provenance are retained beside it.


In [7]:
from IPython.display import FileLink, display
report_html = Path(result['report_html'])
try:
    display(FileLink(str(report_html.relative_to(ROOT))))
except ValueError:
    print('Report outside the Jupyter root; open it locally:', report_html)
print('Portable report folder:', report_html.parent)


../SO4/report.html

Portable report folder: ../SO4


## 7. Optional automatic report evaluation

Provide an independently curated reference report for the same case. This sample download does not contain
a verified reference report. Do not generate a reference from the model's output or infer it from a TCGA project label.
Use `generated_long_report.txt`, not the entire HTML viewer (which also contains evidence and differential diagnoses).
See [evaluation](docs/EVALUATION.md) for field applicability, metrics, structured references and batch input.
The two physician scoring tools are described separately in [scoring](docs/SCORING.md).


In [8]:
REFERENCE_REPORT = None  # Path('/your/reference_report.txt')
if REFERENCE_REPORT is None:
    print('Evaluation skipped: supply an independent reference report first.')
else:
    from evaluation.run_evaluation import evaluate_paths
    metrics = evaluate_paths(Path(result['report_text']), Path(REFERENCE_REPORT))
    evaluation_path = Path(result['run_dir']) / 'evaluation_v4.json'
    evaluation_path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False) + '\n')
    print('Evaluation saved:', evaluation_path)


Evaluation skipped: supply an independent reference report first.


## Saved generated report

The following text was copied from this run's `generated_long_report.txt`.

```text
Specimen/Site:
Adrenal gland / adrenocortical tissue

Diagnosis:
adrenocortical carcinoma

Tumor/Lesion Description:
The examined tissue demonstrates a solid, expansive neoplasm composed of diffuse sheets and solid nests of cells with focal trabecular arrangement. The tumor is composed of large polygonal cells with abundant eosinophilic to clear vacuolated (lipidized) cytoplasm. Segmentation analysis indicates a high neoplastic tissue fraction (up to 98.9%).

Key Microscopic Findings:
Microscopic evaluation reveals a malignant neoplasm characterized by significant cytologic atypia. Tumor cells exhibit marked nuclear pleomorphism with enlarged, vesicular nuclei and prominent nucleoli. The cells are arranged in diffuse sheets and solid nests. Scant stroma is present, with focal hemorrhage and congestion noted.

Invasion/Necrosis/Margins:
Invasion: Not definitively assessable in these ROIs; tumor appears solid and expansive.
Necrosis: Present in focal areas within the provided fields of view.
Margins: Not assessable from the provided images.

Immunohistochemistry/Ancillary Studies:
Not assessable from H&E ROI evidence. Recommended stains to confirm adrenocortical origin include Inhibin-alpha, Melan-A, SF-1, and Calretinin. CK7 and HepPar-1 are recommended to exclude metastasis.

Stage/Grade/Prognostic Features:
Malignancy: malignant based on the locked final diagnosis; grade: high.
Grade: High
The tumor exhibits features of high-grade malignancy, including significant nuclear atypia and pleomorphism. A formal Weiss score assessment requires full slide review for all 9 criteria and is not possible based on the limited ROI evidence provided.
Subtype: adrenocortical carcinoma.
Invasion: not definitively assessable in these ROIs; tumor appears solid and expansive.
Necrosis: not supported by available ROI evidence.

Final Summary:
The histologic features are consistent with a high-grade adrenocortical carcinoma. The differential diagnosis includes metastatic renal cell carcinoma and metastatic hepatocellular carcinoma; however, the cytologic atypia and solid growth pattern favor a primary adrenocortical malignancy over an adenoma. Confirmation of adrenocortical origin via immunohistochemistry (Melan-A, SF-1, Inhibin-alpha) is recommended. Clinical correlation regarding hormone excess symptoms and tumor size is advised.
```
